### **The Chat Format**
#### **Setup**

In [2]:
import os, openai
from dotenv import load_dotenv, find_dotenv
from langchain_openai import ChatOpenAI
from pprint import pprint


_ = load_dotenv(find_dotenv())
openai.api_key = os.environ['OPENAI_API_KEY']

llm_model = 'gpt-3.5-turbo'
chat = ChatOpenAI(model = llm_model)

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

messages = [
    SystemMessage(content = 'You are an assistant that speaks like Shakespeare.'),
    HumanMessage(content = 'Tell me a joke.'),
    AIMessage(content = 'Why did the chicken cross the road? To get to the other side, good sir!'),
    HumanMessage(content = "I don't know")
]

response = chat.invoke(messages)
pprint(response.content)

In [4]:
messages =  [  
    SystemMessage(content = 'You are a friendly assistant.'),
    HumanMessage(content = 'Hi, my name is Isa')
]

response = chat.invoke(messages, temperature = 1)
pprint(response.content)

print()
messages =  [  
    SystemMessage(content = 'You are a friendly assistant.'),
    HumanMessage(content = 'Can you remind me, What is my name?')
]

response = chat.invoke(messages, temperature = 1)
pprint(response.content)

print()
messages =  [  
    SystemMessage(content = 'You are a friendly assistant.'),
    HumanMessage(content = 'Hi, my name is Isa'),
    AIMessage(content = 'Hi Isa! It\'s nice to meet you. Is there anything I can help you with today?'),
    HumanMessage(content = 'Can you remind me, What is my name?')
]

response = chat.invoke(messages, temperature = 1)
pprint(response.content)

"Hello Isa! It's nice to meet you. How can I assist you today?"

("I'm sorry, but I don't have access to your personal information. You would "
 'need to provide me with your name for me to know it. How may I assist you '
 'today?')

'Your name is Isa! How can I assist you further, Isa?'


#### **OrderBot**

In [ ]:
#!pip install jupyter_bokeh
import warnings
import logging
warnings.filterwarnings("ignore")
logging.getLogger("bokeh").setLevel(logging.ERROR)

import panel as pn

# Initialize Panel extension
pn.extension()

# Message context using LangChain's message classes
context = [ SystemMessage(content = '''
You are OrderBot, an automated service to collect orders for a pizza restaurant.
You first greet the customer, then collect the order,
and then ask if it's a pickup or delivery.
You wait to collect the entire order, then summarize it and check for a final
time if the customer wants to add anything else.
If it's a delivery, you ask for an address.
Finally you collect the payment.
Make sure to clarify all options, extras and sizes to uniquely
identify the item from the menu.
You respond in a short, very conversational friendly style.
The menu includes
pepperoni pizza  12.95, 10.00, 7.00
cheese pizza   10.95, 9.25, 6.50
eggplant pizza   11.95, 9.75, 6.75
fries 4.50, 3.50
greek salad 7.25
Toppings:
extra cheese 2.00,
mushrooms 1.50
sausage 3.00
canadian bacon 3.50
AI sauce 1.50
peppers 1.00
Drinks:
coke 3.00, 2.00, 1.00
sprite 3.00, 2.00, 1.00
bottled water 5.00
''')]

# Display panels list
panels = []

# Widgets for user input and interaction
inp = pn.widgets.TextInput(value = 'Hi', placeholder = 'Enter text here..')
button_conversation = pn.widgets.Button(name = 'Chat!')

def collect_messages(_):
    prompt = inp.value_input
    inp.value = ''
    context.append(HumanMessage(content = prompt))

    response = chat.invoke(context)
    # The response is usually an AIMessage; get the .content
    response_text = getattr(response, 'content', str(response))
    context.append(AIMessage(content = response_text))

    panels.append(
        pn.Row('User:', pn.pane.Markdown(prompt, width = 600))
    )
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response_text, width = 600))
    )
    return pn.Column(*panels)

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator = True, height = 300)
)

dashboard  # Show in Jupyter Notebook

In [ ]:
# ---- ORDER SUMMARY (add this in a new cell after you're done chatting) ----
messages = context.copy()
messages.append( SystemMessage(content = (
            'create a json summary of the previous food order. '
            'Itemize the price for each item. '
            'The fields should be 1) pizza, include size 2) list of toppings '
            '3) list of drinks, include size 4) list of sides include size 5) total price'
        )
    )
)

response = chat.invoke(messages)
pprint(getattr(response, 'content', str(response)))